In [ ]:
!pip install ultralytics onnx onnxruntime-gpu
!pip install tensorrt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -U torchvision --force-reinstall --no-deps

  Using cached torchvision-0.29.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (5.7 kB)
Using cached torchvision-0.29.0-cp313-cp313-manylinux_2_28_x86_64.whl (7.4 MB)
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.29.0
    Uninstalling torchvision-0.29.0:
      Successfully uninstalled torchvision-0.29.0


In [ ]:
import torch
print(torch.__version__)

2.14.0+cu130


In [ ]:
!pip install gradio onnxruntime

In [ ]:
import os, shutil, csv

base = '/content/FracAtlas/FracAtlas'
img_dirs = [f'{base}/images/Fractured', f'{base}/images/Non_fractured']
label_dir = f'{base}/Annotations/YOLO'
splits_dir = f'{base}/Utilities/Fracture Split'

out_base = '/content/yolo_dataset'

for split in ['train', 'valid', 'test']:
    os.makedirs(f'{out_base}/images/{split}', exist_ok=True)
    os.makedirs(f'{out_base}/labels/{split}', exist_ok=True)

    with open(f'{splits_dir}/{split}.csv') as f:
        reader = csv.reader(f)
        next(reader)  # skip header
        image_ids = [row[0] for row in reader]

    copied = 0
    for img_id in image_ids:
        # find the image in either Fractured or Non_fractured folder
        img_path = None
        for d in img_dirs:
            candidate = os.path.join(d, img_id)
            if os.path.exists(candidate):
                img_path = candidate
                break

        label_id = img_id.replace('.jpg', '.txt')
        label_path = os.path.join(label_dir, label_id)

        if img_path and os.path.exists(label_path):
            shutil.copy(img_path, f'{out_base}/images/{split}/{img_id}')
            shutil.copy(label_path, f'{out_base}/labels/{split}/{label_id}')
            copied += 1

    print(f'{split}: {copied}/{len(image_ids)} matched with labels')

train: 574/574 matched with labels
valid: 82/82 matched with labels
test: 63/63 matched with labels


In [ ]:
yaml_content = """path: /content/yolo_dataset
train: images/train
val: images/valid
test: images/test

names:
  0: fractured
"""

with open('/content/data_baseline.yaml', 'w') as f:
    f.write(yaml_content)

print(yaml_content)

path: /content/yolo_dataset
train: images/train
val: images/valid
test: images/test

names:
  0: fractured



In [ ]:
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/fracture_weights/baseline_seed7_best.pt')

model.export(format='onnx', imgsz=1024, dynamic=False, simplify=True)

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.14.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 20.7 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/fracture_weights/baseline_seed7_best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 5, 21504) (6.0 MB)

ONNX: starting export with onnx 1.21.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 1.9s, saved as '/content/drive/MyDrive/fracture_weights/baseline_seed7_best.onnx' (11.9 MB)

Export complete (2.9s)
Results saved to /content/drive/MyDrive/fracture_weights/baseline_seed7_best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/fracture_weights/baseline_seed7_best.onnx imgsz=1024 
Validate:        yolo val task=detect model=/content/drive/MyDrive/fracture_weights/baseline_seed7_best.onnx imgsz=1024 data=/content/data_baseline.yaml  
Visualize:       https://netron.app


'/content/drive/MyDrive/fracture_weights/baseline_seed7_best.onnx'

In [ ]:
model.export(format='engine', imgsz=1024, half=True)

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ TensorRT requires GPU export, automatically assigning device=0
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.14.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 20.7 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/fracture_weights/baseline_seed7_best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 5, 21504) (6.0 MB)

ONNX: starting export with onnx 1.21.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 0.9s, saved as '/content/drive/MyDrive/fracture_weights/baseline_seed7_best.onnx' (11.9 MB)

TensorRT: starting export with TensorRT 11.2.1.2...
TensorRT: converting ONNX to FP16 mixed precision with ModelOpt AutoCast...
2026-09-03 10:10:21,542 - [modelopt][onnx] - WARNING - Shared constants were detected and duplicated accordingly.


2026-09-03 10:10:21,735 - [modelopt][onnx] - INFO - Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


INFO:modelopt.onnx:Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


2026-09-03 10:10:22,560 - [modelopt][onnx] - INFO - Running ONNX Runtime to obtain reference outputs (this may take a while)...


INFO:modelopt.onnx.autocast:Running ONNX Runtime to obtain reference outputs (this may take a while)...


...

2026-09-03 10:10:26,588 - [modelopt][onnx] - WARNING - Some initializers contain values smaller than smallest fp16 value, values will be replaced with 6.0e-08.


2026-09-03 10:10:28,134 - [modelopt][onnx] - INFO - Converted 223/231 nodes (96.54%) to fp16


INFO:modelopt.onnx.autocast:Converted 223/231 nodes (96.54%) to fp16


TensorRT: input "images" with shape(1, 3, 1024, 1024) DataType.FLOAT
TensorRT: output "output0" with shape(1, 5, 21504) DataType.FLOAT
TensorRT: building FP16 engine as /content/drive/MyDrive/fracture_weights/baseline_seed7_best.engine
TensorRT: export success ✅ 148.6s, saved as '/content/drive/MyDrive/fracture_weights/baseline_seed7_best.engine' (107.7 MB)

Export complete (149.5s)
Results saved to /content/drive/MyDrive/fracture_weights/baseline_seed7_best.engine
Predict:         yolo predict task=detect model=/content/drive/MyDrive/fracture_weights/baseline_seed7_best.engine imgsz=1024 quantize=16
Validate:        yolo val task=detect model=/content/drive/MyDrive/fracture_weights/baseline_seed7_best.engine imgsz=1024 data=/content/data_baseline.yaml quantize=16 
Visualize:       https://netron.app


PosixPath('/content/drive/MyDrive/fracture_weights/baseline_seed7_best.engine')

In [ ]:
!yolo val task=detect model='/content/drive/MyDrive/fracture_weights/baseline_seed7_best.engine' imgsz=1024 data='/content/data_baseline.yaml'

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.14.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/fracture_weights/baseline_seed7_best.engine for TensorRT inference...
[09/03/2026-10:13:18] [TRT] [I] Loaded engine size: 107 MiB
[09/03/2026-10:13:18] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +32, now: CPU 0, GPU 140 (MiB)
Setting batch=1 input of shape (1, 3, 1024, 1024)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 869.8±350.5 MB/s, size: 16.2 KB)
val: Scanning /content/yolo_dataset/labels/valid.cache... 82 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 82/82 19.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 82/82 54.6it/s 1.5s
                   all         82         91      0.664      0.505      0.496      0.241
Speed: 3.4ms preprocess, 4.7ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/runs/detect/

In [ ]:
import time

def benchmark(model_path, task_name, imgs_path, runs=50):
    m = YOLO(model_path)
    # warmup
    for _ in range(5):
        m.predict(imgs_path, verbose=False)
    start = time.time()
    for _ in range(runs):
        m.predict(imgs_path, verbose=False)
    avg_ms = (time.time() - start) / runs * 1000
    print(f"{task_name}: {avg_ms:.2f} ms")
    return avg_ms

test_img = '/content/FracAtlas/FracAtlas/images/Fractured/IMG0002389.jpg'

pt_time = benchmark('/content/drive/MyDrive/fracture_weights/baseline_seed7_best.pt', 'PyTorch', test_img)
onnx_time = benchmark('/content/drive/MyDrive/fracture_weights/baseline_seed7_best.onnx', 'ONNX', test_img)
trt_time = benchmark('/content/drive/MyDrive/fracture_weights/baseline_seed7_best.engine', 'TensorRT FP16', test_img)

PyTorch: 16.13 ms
Loading /content/drive/MyDrive/fracture_weights/baseline_seed7_best.onnx for ONNX Runtime inference...
WARNING ⚠️ CUDA requested but CUDAExecutionProvider not available. Using CPU...
Using ONNX Runtime 1.29.0 with CPUExecutionProvider
ONNX: 405.87 ms
Loading /content/drive/MyDrive/fracture_weights/baseline_seed7_best.engine for TensorRT inference...
TensorRT FP16: 12.12 ms
